In [1]:
# Run the for loop in each function to return a dictionary
# Make a seperate function that turns each dictionary into a database 

In [2]:
from datetime import datetime
from dotenv import load_dotenv
import pandas as pd
import requests
import sqlite3
import os
import yfinance as yf

In [3]:
load_dotenv()

True

In [4]:
fmg_api_key = os.getenv("FMG_API_KEY")
tiingo_key = os.getenv("TIINGO_TOKEN")

In [5]:
symbols_string = "AAPL, TSLA, AMZN, MSFT, NVDA, GOOGL, META, NFLX, JPM, V, BAC, PYPL, DIS, T, PFE, COST, INTC, KO, TGT, NKE, BA, BABA, XOM, WMT, GE, CSCO, VZ, JNJ, CVX, PLTR, SQ, SHOP, SBUX, SOFI, HOOD, RBLX, SNAP, AMD, UBER, FDX, ABBV, ETSY, MRNA, LMT, GM, F, LCID, CCL, DAL, UAL, AAL, TSM, SONY, ET, MRO, COIN, RIVN, RIOT, CPRX, NOK, ROKU, VIAC, ATVI, BIDU, DOCU, ZM, PINS, TLRY, WBA, MGM, NIO, C, GS, WFC, ADBE, PEP, UNH, CARR, HCA, TWTR, BILI, SIRI, FUBO, RKT"
symbols_split = symbols_string.split(',')
symbols = [s.strip() for s in symbols_split]

### Get Financial Statements

In [6]:
def get_income_statement(ticker, key):
    # This function will return the income statement, balance sheet, and cash flow statement of a company
    income_endpoint = f"https://financialmodelingprep.com/stable/income-statement?symbol={ticker}&apikey={key}"
    income_requests = requests.get(income_endpoint)
    income_data = income_requests.json()
    return income_data

def get_balance_sheet(ticker, key):
    balance_endpoint = f"https://financialmodelingprep.com/stable/balance-sheet-statement?symbol={ticker}&apikey={key}"
    balance_requests = requests.get(balance_endpoint)
    balance_data = balance_requests.json()
    return balance_data

def get_cash_flow_statement(ticker, key):
    cash_endpoint = f"https://financialmodelingprep.com/stable/cash-flow-statement?symbol={ticker}&apikey={key}"
    cash_requests = requests.get(cash_endpoint)
    cash_data = cash_requests.json()
    return cash_data

def get_income_growth(ticker, key):
    income_growth_endpoint = f"https://financialmodelingprep.com/stable/income-statement-growth?symbol={ticker}&apikey={key}"
    income_growth_requests = requests.get(income_growth_endpoint)
    income_growth_data = income_growth_requests.json()
    return income_growth_data

### Price History

In [7]:
def yfinance_historic_price(ticker):
    yfinance_ticker = yf.Ticker(ticker)
    price_history = yfinance_ticker.history('5y')
    price_history = price_history.reset_index()

    return price_history

### Share History

In [8]:
def yfinance_get_shares(ticker):
    yfinance_ticker = yf.Ticker(ticker)
    shares = yfinance_ticker.get_shares_full(start='2021-10-01')
    df_shares = pd.DataFrame(data=shares,  index=None,)
    df_shares.reset_index(inplace=True)
    df_shares = df_shares.rename(columns={'index': 'Date', 0: 'Shares'})
    df_shares['Date'] = pd.to_datetime(df_shares['Date']).dt.date

    return df_shares

In [9]:
def get_fiscal_years(ticker, key):
    income_statement = get_income_statement(ticker=ticker, key=key)

    fiscal_years = []
    for year in income_statement:
        fiscal_year = year['fiscalYear']
        fiscal_years.append(fiscal_year)

    return fiscal_years

In [10]:
fiscal_years = get_fiscal_years(ticker='AAPL', key=fmg_api_key)
fiscal_years_dict = {'Fiscal Year': fiscal_years}

In [11]:
test_stocks = ['AAPL', 'TSLA', 'AMZN']

### Market Cap

In [12]:
balance_sheet = get_balance_sheet(key=fmg_api_key, ticker='AAPL')

In [13]:
def calc_market_cap(ticker, balance):
    market_caps = []

    for year in balance:
        # Find filing date
        file_date = year['filingDate']
        
        # Find price on filing date
        price_history = yfinance_historic_price(ticker=ticker)
        filing_date_stock_price = price_history[price_history['Date'] == file_date]
        
        # Convert file date to datetime for next step
        file_date = pd.to_datetime(file_date)
        # Find closest date on shares df
        shares_history = yfinance_get_shares(ticker=ticker)
        shares_history['Date'] = pd.to_datetime(shares_history['Date'])
        closest_date = (shares_history['Date'] - file_date).abs().idxmin()
        closest_row = shares_history.loc[closest_date]
        # Calculate Market Cap
        market_cap = closest_row['Shares'] * filing_date_stock_price['Close'].iloc[0]
        market_cap = float(market_cap)
        market_caps.append(market_cap)

    return market_caps

In [14]:
market_cap_list = []
for stock in test_stocks:
    balance_sheet = get_balance_sheet(ticker=stock, key=fmg_api_key)
    market_caps = calc_market_cap(ticker=stock, balance=balance_sheet)
    market_cap_list.append(market_caps)

market_cap_list

[[4001076570162.964,
  3417737538413.9062,
  2727276327016.172,
  2432145721066.0312,
  2399216450244.1406],
 [1563113058093.829,
  1284922825690.4297,
  608071934428.7812,
  548085422491.5469,
  312581647530.7969],
 [2257768636003.3086,
  2409512259149.5312,
  1784659113660.9375,
  1059468350670.9844,
  80213911626.09863]]

In [15]:
market_cap_dict = dict(zip(test_stocks, market_cap_list))
market_cap_dict

{'AAPL': [4001076570162.964,
  3417737538413.9062,
  2727276327016.172,
  2432145721066.0312,
  2399216450244.1406],
 'TSLA': [1563113058093.829,
  1284922825690.4297,
  608071934428.7812,
  548085422491.5469,
  312581647530.7969],
 'AMZN': [2257768636003.3086,
  2409512259149.5312,
  1784659113660.9375,
  1059468350670.9844,
  80213911626.09863]}

In [16]:
df_market_cap = pd.DataFrame.from_dict(market_cap_dict)
df_market_cap

,AAPL,TSLA,AMZN
0,4.001077e+12,1.563113e+12,2.257769e+12
1,3.417738e+12,1.284923e+12,2.409512e+12
2,2.727276e+12,6.080719e+11,1.784659e+12
3,2.432146e+12,5.480854e+11,1.059468e+12
4,2.399216e+12,3.125816e+11,8.021391e+10


### P/B ratio function

In [17]:
def get_shareholder_equity(ticker, balance):
    shareholder_equity = []
    for year in balance:
        equity = year['totalStockholdersEquity']
        shareholder_equity.append(equity)

    return shareholder_equity

In [18]:
s_equity_list = []
for stock in test_stocks:
    balance_sheet = get_balance_sheet(ticker=stock, key=fmg_api_key)
    equity = get_shareholder_equity(ticker=stock, balance=balance_sheet)
    s_equity_list.append(equity)
s_equity_list

[[73733000000, 56950000000, 62146000000, 50672000000, 63090000000],
 [82137000000, 72913000000, 62634000000, 44704000000, 30189000000],
 [411065000000, 285970000000, 201875000000, 146043000000, 138245000000]]

In [19]:
s_equity_dict = dict(zip(test_stocks, s_equity_list))
s_equity_dict

{'AAPL': [73733000000, 56950000000, 62146000000, 50672000000, 63090000000],
 'TSLA': [82137000000, 72913000000, 62634000000, 44704000000, 30189000000],
 'AMZN': [411065000000,
  285970000000,
  201875000000,
  146043000000,
  138245000000]}

In [20]:
df_shareholders_equity = pd.DataFrame.from_dict(s_equity_dict)
df_shareholders_equity

,AAPL,TSLA,AMZN
0,73733000000,82137000000,411065000000
1,56950000000,72913000000,285970000000
2,62146000000,62634000000,201875000000
3,50672000000,44704000000,146043000000
4,63090000000,30189000000,138245000000


In [21]:
def calc_pb_ratio(market_cap, equity):
    return round(market_cap / equity, 2)

df_pb_ratio = calc_pb_ratio(market_cap=df_market_cap, equity=df_shareholders_equity)
df_pb_ratio

,AAPL,TSLA,AMZN
0,54.26,19.03,5.49
1,60.01,17.62,8.43
2,43.88,9.71,8.84
3,48.00,12.26,7.25
4,38.03,10.35,0.58


In [22]:
df_pb_ratio.insert(loc=0, column='Fiscal Year', value=fiscal_years)
df_pb_ratio

,Fiscal Year,AAPL,TSLA,AMZN
0,2025,54.26,19.03,5.49
1,2024,60.01,17.62,8.43
2,2023,43.88,9.71,8.84
3,2022,48.00,12.26,7.25
4,2021,38.03,10.35,0.58


### Debt to Equity function

In [23]:
def calc_de_ratio(ticker, balance):
    de_ratios = []
    for year in balance:
        # Step 1: Pull shareholder and total debt from balance sheet
        shareholder_equity = year['totalStockholdersEquity']
        total_debt = year['totalDebt']
        # Step 2: Divide numbers
        de_ratio = total_debt / shareholder_equity
        # Step 3: Append yearly ratios to list
        de_ratios.append(de_ratio)

    return de_ratios

In [24]:
de_ratios_list = []
for stock in test_stocks:
    balance_sheet = get_balance_sheet(ticker=stock, key=fmg_api_key)
    de_ratio = calc_de_ratio(ticker=stock, balance=balance_sheet)
    de_ratios_list.append(de_ratio)

de_ratios_list

[[1.5241072518411023,
  2.090588235294118,
  1.9941750072410132,
  2.6144616356173036,
  2.163924552226977],
 [0.1019759669819935,
  0.18683910962379824,
  0.15284031037455695,
  0.12857909806728704,
  0.2939150021531021],
 [0.37217228418863196,
  0.45774032241144175,
  0.6717572755417957,
  0.9594297569893798,
  0.8419472675322797]]

In [25]:
de_ratios_dict = dict(zip(test_stocks, de_ratios_list))
de_ratios_dict

{'AAPL': [1.5241072518411023,
  2.090588235294118,
  1.9941750072410132,
  2.6144616356173036,
  2.163924552226977],
 'TSLA': [0.1019759669819935,
  0.18683910962379824,
  0.15284031037455695,
  0.12857909806728704,
  0.2939150021531021],
 'AMZN': [0.37217228418863196,
  0.45774032241144175,
  0.6717572755417957,
  0.9594297569893798,
  0.8419472675322797]}

In [26]:
de_ratio_full = fiscal_years_dict | de_ratios_dict
de_ratio_full

{'Fiscal Year': ['2025', '2024', '2023', '2022', '2021'],
 'AAPL': [1.5241072518411023,
  2.090588235294118,
  1.9941750072410132,
  2.6144616356173036,
  2.163924552226977],
 'TSLA': [0.1019759669819935,
  0.18683910962379824,
  0.15284031037455695,
  0.12857909806728704,
  0.2939150021531021],
 'AMZN': [0.37217228418863196,
  0.45774032241144175,
  0.6717572755417957,
  0.9594297569893798,
  0.8419472675322797]}

In [27]:
df_de_ratio = pd.DataFrame.from_dict(de_ratio_full)
df_de_ratio

,Fiscal Year,AAPL,TSLA,AMZN
0,2025,1.524107,0.101976,0.372172
1,2024,2.090588,0.186839,0.457740
2,2023,1.994175,0.152840,0.671757
3,2022,2.614462,0.128579,0.959430
4,2021,2.163925,0.293915,0.841947


### Revenue Growth

In [28]:
def calc_revenue_growth(ticker, income_growth):
    revenue_growth = []
    for year in income_growth:
        growth = year['growthRevenue']
        revenue_growth.append(growth)

    return revenue_growth

In [29]:
revenue_growth_list = []
for stock in test_stocks:
    income_growth_statement = get_income_growth(ticker=stock, key=fmg_api_key)
    revenue_growth = calc_revenue_growth(ticker=stock, income_growth=income_growth_statement)
    revenue_growth_list.append(revenue_growth)

In [30]:
revenue_growth_dict = dict(zip(test_stocks, revenue_growth_list))

In [31]:
revenue_growth_full = fiscal_years_dict | revenue_growth_dict

In [32]:
df_revenue_growth = pd.DataFrame.from_dict(data=revenue_growth_full)
df_revenue_growth

,Fiscal Year,AAPL,TSLA,AMZN
0,2025,0.064255,-0.029307,0.123778
1,2024,0.020220,0.009476,0.109909
2,2023,-0.028005,0.187953,0.118296
3,2022,0.077938,0.513517,0.093995
4,2021,0.332594,0.706716,0.216954


### Gross Profit Margins

In [33]:
def calc_gross_profit_margin(ticker, income):
    gross_profit_margin = []
    for year in income:
        total_revenue = year['revenue']
        gross_profit = year['grossProfit']
        gpm = gross_profit / total_revenue
        gross_profit_margin.append(gpm)

    return gross_profit_margin

In [34]:
gpm_list = []

for stock in test_stocks:
    income_statement = get_income_statement(ticker=stock, key=fmg_api_key)
    gross_profit_margin = calc_gross_profit_margin(ticker=stock, income=income_statement)
    gpm_list.append(gross_profit_margin)

In [35]:
gpm_dict = dict(zip(test_stocks, gpm_list))

In [36]:
gpm_full = fiscal_years_dict | gpm_dict

In [37]:
df_gross_profit_margin = pd.DataFrame.from_dict(gpm_full)
df_gross_profit_margin

,Fiscal Year,AAPL,TSLA,AMZN
0,2025,0.469052,0.180265,0.502857
1,2024,0.462063,0.178626,0.488544
2,2023,0.441311,0.182489,0.469821
3,2022,0.433096,0.255984,0.438053
4,2021,0.417794,0.252792,0.420325


### Altman Z-score

In [38]:
# industries = []
# skipped = []
# for stock in symbols:
#     yf_ticker = yf.Ticker(stock)
#     company_info = yf_ticker.info
#     try:
#         industry = company_info['industry']
#     except:
#         skipped.append(stock)
#         continue
#     industries.append(industry)
# # industries

In [39]:
software_ind = ['Software - Application', 'Internet Content & Information',]

In [40]:
# Altman Z-Score = 1.2A + 1.4B + 3.3C + 0.6D + 1.0E

# Where:

# A = working capital (current assets - current liabilities) / total assets
# B = retained earnings / total assets
# C = earnings before interest and tax / total assets
# D = market value of equity / total liabilities
# E = sales / total assets

In [41]:
def get_z_balance_inputs(ticker):
    balance_sheet = get_balance_sheet(ticker=ticker, key=fmg_api_key)
    total_assets_list = []
    current_assets_list = []
    total_liabilities_list = []
    current_liabilities_list = []
    retained_earnings_list = []
    for year in balance_sheet:
        total_assets = year['totalAssets']
        total_assets_list.append(total_assets)
        
        current_assets = year['totalCurrentAssets']
        current_assets_list.append(current_assets)
        
        total_liabilities = year['totalLiabilities']
        total_liabilities_list.append(total_liabilities)
        
        current_liabilities = year['totalCurrentLiabilities']
        current_liabilities_list.append(current_liabilities)

        retained_earnings = year['retainedEarnings']
        retained_earnings_list.append(retained_earnings)

    return total_assets_list, current_assets_list, total_liabilities_list, current_liabilities_list, retained_earnings_list
        
def get_z_income_inputs(ticker):
    income_statement = get_income_statement(ticker=ticker, key=fmg_api_key)
    ebit_list = []
    sales_list = []
    for year in income_statement:
        ebit = year['ebit']
        ebit_list.append(ebit)

        sales = year['revenue']
        sales_list.append(sales)

    return ebit_list, sales_list

In [42]:
total_assets = []
current_assets = []
total_liabilities = []
current_liabilities = []
retained_earnings = []
ebit = []
sales = []

In [43]:
for stock in test_stocks:
    total_a, current_a, total_l, current_l, retained_e = get_z_balance_inputs(ticker=stock,)
    total_assets.append(total_a)
    current_assets.append(current_a)
    total_liabilities.append(total_l)
    current_liabilities.append(current_l)
    retained_earnings.append(retained_e)

    e, s = get_z_income_inputs(ticker=stock)
    ebit.append(e)
    sales.append(s)

In [44]:
metric_names = ['Total Assets', 'Current Assets', 'Total Liabilities', 'Current Liabilities', 'Retained Earnings',
                'ebit', 'sales']
z_score_metrics = [total_assets, current_assets, total_liabilities, current_liabilities, retained_earnings,
                   ebit, sales]

In [45]:
z_score_df = {}
# Combine z_score metrics into one dictionary
for name, metric in zip(metric_names, z_score_metrics):
    metric_dict = dict(zip(test_stocks, metric))
    z_score_df[name] = pd.DataFrame.from_dict(metric_dict)

In [46]:
z_score_df['Total Assets']

,AAPL,TSLA,AMZN
0,359241000000,137806000000,818042000000
1,364980000000,122070000000,624894000000
2,352583000000,106618000000,527854000000
3,352755000000,82338000000,462675000000
4,351002000000,62131000000,420549000000


In [47]:
# Next step: Calculate z-score based on manufacturing/non-manufacturing

In [48]:
def get_industry(ticker):
    yf_ticker = yf.Ticker(ticker)
    company_info = yf_ticker.info
    company_ind = company_info['industry']

    return company_ind